In [2]:
pip install langgraph langchain-groq pydantic chromadb sentence-transformers python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv

# --- 1. Print Setup Header ---
print("=== Step 4 Phase 1: Environment & File Setup Check ===")

# --- 2. Check File Existence ---
env_exists = os.path.exists(".env")
print(f"1. '.env' file present: {env_exists}")

data_folder_exists = os.path.exists("data")
print(f"2. 'data/' folder present: {data_folder_exists}")

policy_file_exists = os.path.exists("data/procurement_policy.md")
print(f"3. 'data/procurement_policy.md' present: {policy_file_exists}")

chroma_folder_exists = os.path.exists("./chroma_db")
print(f"4. './chroma_db' folder present: {chroma_folder_exists}")

# --- 3. Load API Key ---
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

key_is_valid = bool(api_key and api_key.startswith("gsk_"))
print(f"5. Valid GROQ_API_KEY loaded: {key_is_valid}")

# --- 4. Final Verdict ---
print("\n--- Final System Readiness Status ---")
if env_exists and policy_file_exists and chroma_folder_exists and key_is_valid:
    print("SUCCESS: Aapka poora environment aur files 100% READY hain!")
    print("Ab aap Step 4 ka main Multi-Agent Code bina kisi error ke chala sakte hain.")
else:
    print("WARNING: Kuch items missing hain! Niche diye gaye steps follow karein:")
    if not env_exists:
        print(" - Fix: Project folder me `.env` file banayein.")
    if not policy_file_exists:
        print(" - Fix: 'data/' folder me 'procurement_policy.md' file rakhein.")
    if not chroma_folder_exists:
        print(" - Fix: Step 2 wala code run karke ChromaDB index create karein.")
    if not key_is_valid:
        print(" - Fix: `.env` file me valid `GROQ_API_KEY=\"gsk_...\"` add karein.")

=== Step 4 Phase 1: Environment & File Setup Check ===
1. '.env' file present: True
2. 'data/' folder present: True
3. 'data/procurement_policy.md' present: True
4. './chroma_db' folder present: True
5. Valid GROQ_API_KEY loaded: True

--- Final System Readiness Status ---
SUCCESS: Aapka poora environment aur files 100% READY hain!
Ab aap Step 4 ka main Multi-Agent Code bina kisi error ke chala sakte hain.


In [8]:
import os
from typing import List, Optional, TypedDict
from dotenv import load_dotenv

from pydantic import BaseModel, Field
import chromadb
from chromadb.utils import embedding_functions

from langchain_groq import ChatGroq
from langchain_core.output_parsers import PydanticOutputParser
from langgraph.graph import StateGraph, START, END

# Load environment configuration
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

# --- 1. Define Pydantic Schemas ---
class InvoiceItem(BaseModel):
    description: str = Field(description="Description of the goods or services")
    quantity: int = Field(description="Quantity of items purchased")
    unit_price: float = Field(description="Unit price per item")
    line_total: float = Field(description="Total price for this line item")

class InvoiceSchema(BaseModel):
    vendor_name: str = Field(description="Name of the selling entity/vendor")
    po_number: Optional[str] = Field(None, description="Associated Purchase Order number, if present")
    invoice_date: str = Field(description="Date invoice was issued")
    invoice_total: float = Field(description="Final total monetary amount")
    has_tax_id: bool = Field(description="True if Tax ID / GSTIN / VAT number is present")
    payment_terms: str = Field(default="Net-30", description="Extracted payment terms e.g. Net-30, Net-15, Immediate")
    line_items: List[InvoiceItem] = Field(default_factory=list, description="List of itemized purchase lines")

class AuditResult(BaseModel):
    is_compliant: bool = Field(description="True if invoice satisfies all policy checks")
    flagged_violations: List[str] = Field(default_factory=list, description="List of policy violations found")
    risk_score: str = Field(description="LOW, MEDIUM, or HIGH risk rating based on audit findings")

class ActionPayload(BaseModel):
    recommended_action: str = Field(description="Action e.g. AUTO_APPROVE, FLAG_FOR_REVIEW, REJECT")
    email_draft: str = Field(description="Drafted response email to vendor or procurement lead")
    ticket_summary: str = Field(description="Concise single-line audit summary for internal tracking")

# --- 2. Initialize Pydantic Parsers ---
extraction_parser = PydanticOutputParser(pydantic_object=InvoiceSchema)
audit_parser = PydanticOutputParser(pydantic_object=AuditResult)
action_parser = PydanticOutputParser(pydantic_object=ActionPayload)

# --- 3. Initialize Vector Store ---
class PolicyVectorStore:
    def __init__(self, persistence_dir: str = "./chroma_db"):
        self.chroma_client = chromadb.PersistentClient(path=persistence_dir)
        self.emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.collection = self.chroma_client.get_or_create_collection(
            name="procurement_policies",
            embedding_function=self.emb_fn
        )

    def query_policy(self, query_text: str, n_results: int = 2) -> list[str]:
        results = self.collection.query(
            query_texts=[query_text],
            n_results=n_results
        )
        return results["documents"][0] if results["documents"] else []

# --- 4. Initialize LLM & Resources ---
selected_model = "openai/gpt-oss-20b"
llm = ChatGroq(
    model_name=selected_model,
    groq_api_key=api_key,
    temperature=0.0
)

vector_store = PolicyVectorStore()

# --- 5. Shared State ---
class CopilotState(TypedDict):
    raw_invoice_text: str
    po_limit: float
    extracted_invoice: InvoiceSchema
    audit_report: AuditResult
    final_action: ActionPayload

# --- 6. Agent Nodes with Explicit Format Instructions ---

def extraction_agent(state: CopilotState) -> dict:
    prompt = f"""
You are a precision data extraction system. Extract billing details from the raw invoice text below.

{extraction_parser.get_format_instructions()}

Invoice Text:
{state['raw_invoice_text']}
"""
    raw_response = llm.invoke(prompt)
    extracted_data = extraction_parser.parse(raw_response.content)
    return {"extracted_invoice": extracted_data}

def policy_audit_agent(state: CopilotState) -> dict:
    inv = state["extracted_invoice"]
    po_limit = state.get("po_limit", inv.invoice_total)

    retrieved_policies = vector_store.query_policy(
        f"Invoice total {inv.invoice_total} vs PO limit {po_limit}, tax ID rules, payment terms {inv.payment_terms}"
    )
    context_str = "\n---\n".join(retrieved_policies)

    prompt = f"""
You are an automated procurement compliance auditor. Compare invoice details against policy rules.

{audit_parser.get_format_instructions()}

Relevant Procurement Policies:
{context_str}

Extracted Invoice Details:
- Vendor: {inv.vendor_name}
- Invoice Total: ${inv.invoice_total:.2f}
- PO Expected Limit: ${po_limit:.2f}
- Tax ID Present: {inv.has_tax_id}
- Payment Terms: {inv.payment_terms}

Determine compliance, highlight rule violations, and assign a risk level.
"""
    raw_response = llm.invoke(prompt)
    audit_res = audit_parser.parse(raw_response.content)
    return {"audit_report": audit_res}

def action_agent(state: CopilotState) -> dict:
    inv = state["extracted_invoice"]
    audit = state["audit_report"]

    prompt = f"""
You are a procurement resolution officer. Generate an operational action payload and vendor response email.

{action_parser.get_format_instructions()}

Invoice Summary:
- Vendor: {inv.vendor_name}
- Total: ${inv.invoice_total:.2f}
    
Audit Report:
- Compliant: {audit.is_compliant}
- Risk Level: {audit.risk_score}
- Violations: {', '.join(audit.flagged_violations) if audit.flagged_violations else 'None'}

Instructions:
1. Set recommended_action to AUTO_APPROVE if compliant, else FLAG_FOR_REVIEW.
2. Draft a clear, polite professional resolution email addressing discrepancies.
3. Provide a concise single-sentence internal ticket summary.
"""
    raw_response = llm.invoke(prompt)
    action_res = action_parser.parse(raw_response.content)
    return {"final_action": action_res}

# --- 7. Assemble Graph ---
workflow = StateGraph(CopilotState)

workflow.add_node("ExtractionAgent", extraction_agent)
workflow.add_node("PolicyAuditAgent", policy_audit_agent)
workflow.add_node("ActionAgent", action_agent)

workflow.add_edge(START, "ExtractionAgent")
workflow.add_edge("ExtractionAgent", "PolicyAuditAgent")
workflow.add_edge("PolicyAuditAgent", "ActionAgent")
workflow.add_edge("ActionAgent", END)

copilot_app = workflow.compile()
print("Multi-Agent LangGraph System compiled successfully with PydanticParsers!")

# --- 8. Test Pipeline Execution ---
sample_input = {
    "raw_invoice_text": """INVOICE #INV-9082
Vendor: Apex Tech Solutions
Date: 2026-03-12
PO Number: PO-8831

Items Purchased:
1. Enterprise Cloud Subscription (Qty: 1) - $2,250.00
Tax ID / GSTIN: Not Provided
Payment Terms: Immediate

Total Amount Due: $2,250.00
""",
    "po_limit": 2000.00
}

print("\nExecuting Pipeline...")
pipeline_result = copilot_app.invoke(sample_input)

print("\n=== EXECUTION SUCCESSFUL ===")
print("1. Extracted Data:\n", pipeline_result["extracted_invoice"].model_dump())
print("\n2. Audit Report:\n", pipeline_result["audit_report"].model_dump())
print("\n3. Action Payload:\n", pipeline_result["final_action"].model_dump())

Multi-Agent LangGraph System compiled successfully with PydanticParsers!

Executing Pipeline...

=== EXECUTION SUCCESSFUL ===
1. Extracted Data:
 {'vendor_name': 'Apex Tech Solutions', 'po_number': 'PO-8831', 'invoice_date': '2026-03-12', 'invoice_total': 2250.0, 'has_tax_id': False, 'payment_terms': 'Immediate', 'line_items': [{'description': 'Enterprise Cloud Subscription', 'quantity': 1, 'unit_price': 2250.0, 'line_total': 2250.0}]}

2. Audit Report:
 {'is_compliant': False, 'flagged_violations': ['POL_TAX_02: Missing Tax ID for invoice over $1,000', 'POL_VAR_01: Invoice exceeds PO limit by more than 5%'], 'risk_score': 'HIGH'}

3. Action Payload:
 {'recommended_action': 'FLAG_FOR_REVIEW', 'email_draft': 'Subject: Request for Invoice Clarification – Apex Tech Solutions\n\nDear Apex Tech Solutions Team,\n\nThank you for submitting Invoice #2026-07 for the recent services rendered. During our audit, we identified two compliance issues that need to be addressed before we can proceed wi